# Israeli legal eval on Google Colab (free T4): qwen3:14b and gemma4:12b, answers + reasoning logs

Answers the same test as the Kaggle notebooks -- **100 questions** (the stratified sample, seed 2026)
and the **first 5 paralegal cases**, with retrieval over the bulk legal corpus -- once per model:
**qwen3:14b** (thinking on) and **gemma4:12b** (thinking off). **No judging here**: the answers
and logs are for review (send the results zip back to Claude for the judging and troubleshooting).

What each model's run leaves (under `RESULTS_DIR/<run>/<model>/`, on your Google Drive):
- `bulk500/answers_rag.jsonl` -- each answer, with the retrieval plan, the retrieved provisions and repairs;
- `cases/answers_rag.jsonl` -- the 5 work files;
- `llm_trace/*.jsonl` -- **every model call in full**: prompt, reasoning ("thinking"), output, stop reason,
  tokens and timing, tagged with the question/case id;
- `reasoning_report.md` (per call: reasoning + output) and `reasoning_report_with_prompts.md` (+ prompts);
- `judge_free_report.txt/json` -- score.py's report without a judge: multiple choice and yes/no labels only;
- `run.log` -- the scripts' own output (per-item timing, retries, errors).

Plus, for the whole run: `summary.md/json` (per-model counts, timing, cut-offs, other-script words, judge-free
score), `corpus_coverage.md/json`, `environment.txt`, `ollama.log`, and a zip of everything.

## Before running
1. *Runtime -> Change runtime type*: **T4 GPU**.
2. *Secrets* (key icon, left): add **GITHUB_TOKEN** (a token that can read `zananiri/AI-IZ`) and turn on
   *Notebook access*.
3. Put **`legal_corpus_vectordb.zip`** (from `notebooks/kaggle_legal_corpus_build.ipynb`) on your Google
   Drive and set `CORPUS_ZIP` below to its path.
4. *Runtime -> Run all*. Allow Drive access when asked.

## Time, and resuming
Rough estimate (not measured on Colab yet): qwen3:14b with thinking ~1 min per question, gemma4:12b
~20-30 s, each case a few minutes -- about **3-4 hours** for both models. Free Colab sessions can end
early. **Everything is written to Drive as it goes**: run all again with the same `RUN_NAME` and every
step skips the ids it already has (answers that errored or came back empty are redone).

In [ ]:
BRANCH = "main"                                   # the branch to test
MODELS = [                                        # answered in this order; comment one out to run only the other
    {"tag": "qwen3:14b", "thinking": True},       # Qwen3: reasoning on (logged in full)
    {"tag": "gemma4:12b", "thinking": False},     # Gemma 4 12B; thinking off, as in the Gemma 3 runs
]
LIMIT = 100             # questions: a stratified sample across the 9 categories (same as Kaggle); None = all 500
SAMPLE_SEED = 2026      # the same seed as the Kaggle runs -> the same 100 questions
RUN_CASES = True        # the paralegal cases
CASE_LIMIT = 5          # the first N cases of cases_gold.json
TOP_K = 12              # retrieved chunks per question (issue spotting: 16, set in eval_run.py)
CASE_TOP_K = 16         # retrieved chunks per case
CONTEXT_LENGTH = 16384  # tokens per call (Ollama num_ctx): prompt + reasoning + answer
ANSWER_MAX_TOKENS = 6144
CASE_MAX_TOKENS = 6144
REQUEST_TIMEOUT_S = 1800  # per model call; a long thinking answer on a T4 can take many minutes

DRIVE_DIR = "/content/drive/MyDrive/AI-IZ"
CORPUS_ZIP = f"{DRIVE_DIR}/legal_corpus_vectordb.zip"   # uploaded from the Kaggle corpus build
RESULTS_DIR = f"{DRIVE_DIR}/colab_eval"
RUN_NAME = "qwen14b_gemma4_12b"   # the same name resumes an earlier run; a new name starts fresh

## 1. Drive, GPU, code

In [ ]:
import os, pathlib, subprocess, shutil, json, sys, time, zipfile
from google.colab import drive, userdata

drive.mount("/content/drive")
RUN_DIR = pathlib.Path(RESULTS_DIR) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
assert pathlib.Path(CORPUS_ZIP).exists(), f"corpus zip not found at {CORPUS_ZIP}: upload it to Drive or fix CORPUS_ZIP"

def sh(command, log=None, check=True, env=None):
    """Runs a shell command with its output streamed here (and appended to `log`); raises on failure
    when check=True -- unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    out = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")
    return proc.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

REPO = "/content/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("add the GITHUB_TOKEN secret (key icon, left) with notebook access") from exc
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
sh("git log --oneline -1")

## 2. Python environment

A separate Python 3.12 environment (`uv`), so Colab's own Python version and packages don't matter
(the project supports 3.10-3.12). chromadb is pinned to the version that built the corpus, read from the
zip's `_build_info.json`.

In [ ]:
def find_corpus(zip_path, depth=0):
    """(zip, prefix) of the vector store inside `zip_path`: its laws/ and procedural_rules/ Chroma folders
    (chroma.sqlite3) may sit at the top, under one extra folder (a re-zipped download), or in a zip inside
    the zip (a Kaggle "download all"). Prints the contents so a wrong file is easy to spot."""
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        print(f"{pathlib.Path(zip_path).name}: {len(names)} entries, e.g.", names[:8])
        stores = sorted({n[: -len("chroma.sqlite3")] for n in names if n.endswith("chroma.sqlite3")})
        if stores:
            # "laws/" -> prefix "", "legal_corpus_vectordb/laws/" -> prefix "legal_corpus_vectordb/"
            prefixes = {s.rstrip("/").rpartition("/")[0] for s in stores}
            prefix = min(prefixes, key=len)
            return zip_path, (prefix + "/") if prefix else ""
        inner = [n for n in names if n.endswith(".zip") and "vectordb" in n] or [n for n in names if n.endswith(".zip")]
        if inner and depth < 2:
            out = pathlib.Path("/content/corpus_inner") / pathlib.Path(inner[0]).name
            out.parent.mkdir(parents=True, exist_ok=True)
            with z.open(inner[0]) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)
            print("  -> using the zip inside it:", inner[0])
            return find_corpus(out, depth + 1)
    if any(n.endswith(".jsonl") for n in names):
        raise RuntimeError("this zip holds the corpus RECORDS (legal_corpus_jsonl.zip), not the database: upload "
                           "legal_corpus_vectordb.zip from the corpus build's Output instead")
    raise RuntimeError(f"no Chroma database (chroma.sqlite3) in {zip_path}: is this legal_corpus_vectordb.zip?")

CORPUS_SRC, CORPUS_PREFIX = find_corpus(CORPUS_ZIP)
with zipfile.ZipFile(CORPUS_SRC) as z:
    info_name = CORPUS_PREFIX + "_build_info.json"
    BUILD_INFO = json.loads(z.read(info_name)) if info_name in z.namelist() else {}
    CORPUS_CATEGORIES = sorted({n[len(CORPUS_PREFIX):].split("/")[0] for n in z.namelist()
                                if n.startswith(CORPUS_PREFIX) and n.endswith("chroma.sqlite3")})
print("corpus categories:", CORPUS_CATEGORIES)
CHROMADB_VERSION = BUILD_INFO.get("chromadb_version")
if BUILD_INFO:
    print("corpus built with chromadb", CHROMADB_VERSION, "| embedding model", BUILD_INFO.get("embedding_model"))
else:
    print("[note] no _build_info.json in the zip: chromadb isn't pinned (the latest is installed) and chunk counts "
          "aren't checked; if the store won't open, rebuild the zip with the corpus build notebook")

VENV = "/content/venv"
PY = f"{VENV}/bin/python"
if not pathlib.Path(PY).exists():
    sh("pip install -q uv")
    sh(f"uv venv -q --python 3.12 {VENV}")
sh(f'uv pip install -q --python {PY} -e ".[legal,legal-data]"' + (f' "chromadb=={CHROMADB_VERSION}"' if CHROMADB_VERSION else ""))
sh(f"{PY} --version")

## 3. Ollama and the two models

`OLLAMA_MAX_LOADED_MODELS=1`: one model on the GPU at a time. Flash attention + an 8-bit KV cache halve the
16k context's memory. Retrieval (bge-m3 + the reranker) runs on the CPU in step 6, so the T4's 15 GB stay
with the model.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

OLLAMA_LOG = RUN_DIR / "ollama.log"
if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_FLASH_ATTENTION": "1",
                                   "OLLAMA_KV_CACHE_TYPE": "q8_0"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)
for m in MODELS:
    sh(f"ollama pull {m['tag']}")
sh("ollama list")

## 4. The corpus (from Drive) and the eval set (from the repo)

Also checks every law and section the gold answers cite against the installed corpus
(`corpus_coverage.md`) -- a missing law shows up there before it shows up as a wrong answer.

In [ ]:
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
if not all((VECTORDB / c / "chroma.sqlite3").exists() for c in CORPUS_CATEGORIES):
    if CORPUS_PREFIX == "" and BUILD_INFO and CORPUS_SRC == CORPUS_ZIP:
        sh(f'{PY} scripts/legal_data/install_corpus.py "{CORPUS_SRC}" --replace')   # also checks chunk counts
    else:
        # A re-zipped or nested zip: unpack the store itself, without the extra folder level.
        shutil.rmtree(VECTORDB, ignore_errors=True)
        VECTORDB.mkdir(parents=True)
        with zipfile.ZipFile(CORPUS_SRC) as z:
            for member in z.infolist():
                name = member.filename
                if not name.startswith(CORPUS_PREFIX) or name.endswith("/") or ".." in name:
                    continue
                dest = VECTORDB / name[len(CORPUS_PREFIX):]
                dest.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, open(dest, "wb") as dst:
                    shutil.copyfileobj(src, dst)
        print("unpacked", CORPUS_CATEGORIES, "into", VECTORDB)
missing = [c for c in ("laws", "procedural_rules") if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")

EVAL_DIR = pathlib.Path("legal_txt/Evals")
zipfile.ZipFile(EVAL_DIR / "israeli_legal_eval.zip").extractall(EVAL_DIR)
EVAL = EVAL_DIR / "israeli_legal_eval"
assert (EVAL / "questions.jsonl").exists(), f"eval set not found under {EVAL}"

sh(f'{PY} scripts/legal_data/check_corpus_coverage.py --gold {EVAL / "gold.jsonl"} --categories {",".join(CORPUS_CATEGORIES)} '
   f'--out "{RUN_DIR / "corpus_coverage.md"}" --json-out "{RUN_DIR / "corpus_coverage.json"}"', check=False)

## 5. The same 100 questions as the Kaggle runs

A proportional slice of each of the 9 categories, picked with `SAMPLE_SEED` (the Kaggle notebooks' code).

In [ ]:
import random
from collections import Counter, defaultdict

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

all_questions = load_jsonl(EVAL / "questions.jsonl")
QUESTIONS = RUN_DIR / "questions_sample.jsonl"
if LIMIT and LIMIT < len(all_questions):
    by_cat = defaultdict(list)
    for q in all_questions:
        by_cat[q["category"]].append(q)
    share, rng, sample = LIMIT / len(all_questions), random.Random(SAMPLE_SEED), []
    for qs in by_cat.values():
        k = max(1, round(len(qs) * share))
        picked = qs[:k] if SAMPLE_SEED is None else rng.sample(qs, k)
        sample += sorted(picked, key=lambda q: q["id"])
else:
    sample = all_questions
with open(QUESTIONS, "w", encoding="utf-8") as f:
    for q in sample:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(sample)} questions ->", QUESTIONS)
print("by category:", dict(Counter(q["category"] for q in sample).most_common()))

## 6. Answer: each model in turn (resumable)

Each model gets its own config (Ollama, its tag, the 16k context, the timeout) and its own output folder
and trace. Retrieval runs on the CPU (`CUDA_VISIBLE_DEVICES=""` for these scripts only; Ollama keeps the
GPU). Each line of output is `id (category): Ns`.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))

def model_dir(tag):
    return RUN_DIR / tag.replace(":", "_").replace("/", "_")

def model_env(m):
    out = model_dir(m["tag"])
    for sub in ("bulk500", "cases", "llm_trace"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    cfg = json.loads(json.dumps(base_cfg))
    orch = cfg["legal"]["orchestrator"]
    orch.update(backend="ollama", base_url="http://localhost:11434", model=m["tag"],
                max_model_len=CONTEXT_LENGTH, request_timeout_s=REQUEST_TIMEOUT_S)
    cfg.setdefault("logging", {})["llm_trace_dir"] = str(out / "llm_trace")
    cfg_path = out / "config.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return out, {**os.environ, "DOCSLIDES_CONFIG": str(cfg_path), "CUDA_VISIBLE_DEVICES": "",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": m["tag"],
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
                 "DOCSLIDES_LLM_TRACE_DIR": str(out / "llm_trace")}

def drop_failed(path):
    """Resume: remove answers that errored or came back empty, so this run redoes them."""
    if not path.exists():
        return 0
    rows = load_jsonl(path)
    good = [r for r in rows if not r.get("error") and str(r.get("answer") or r.get("work_file") or "").strip()]
    if len(good) < len(rows):
        with open(path, "w", encoding="utf-8") as f:
            for r in good:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(rows) - len(good)

for m in MODELS:
    out, env = model_env(m)
    log = out / "run.log"
    no_think = "" if m["thinking"] else " --no-thinking"
    answers = out / "bulk500" / "answers_rag.jsonl"
    print(f"\n{'=' * 20} {m['tag']}: questions {'=' * 20}  (redoing {drop_failed(answers)} failed)")
    sh(f'{PY} scripts/legal_data/eval_run.py answer --questions "{QUESTIONS}" --top-k {TOP_K} '
       f'--max-tokens {ANSWER_MAX_TOKENS} --out "{answers}"{no_think}', log=log, env=env)
    if RUN_CASES:
        cases = out / "cases" / "answers_rag.jsonl"
        print(f"\n{'=' * 20} {m['tag']}: {CASE_LIMIT} cases {'=' * 20}  (redoing {drop_failed(cases)} failed)")
        sh(f'{PY} scripts/legal_data/eval_cases.py answer --dir {EVAL} --top-k {CASE_TOP_K} '
           f'--max-tokens {CASE_MAX_TOKENS} --limit {CASE_LIMIT} --out "{cases}"{no_think}', log=log, env=env)
    subprocess.run(["ollama", "stop", m["tag"]], capture_output=True)  # free the GPU for the next model

## 7. Reasoning logs, judge-free scores, summary

- `reasoning_report.md`: per question/case, every call's reasoning, output, stop reason and timing;
  `reasoning_report_with_prompts.md` adds the full prompts (retrieved evidence included).
- `judge_free_report.*`: score.py (unmodified) without a judge -- the 16 multiple-choice and 18 yes/no
  label scores only; the judged part waits for review.

In [ ]:
import re, statistics

FOREIGN = re.compile(r"[Ѐ-ӿ؀-ۿ一-鿿぀-ヿ가-힯฀-๿]+|[A-Za-z]{4,}")
summary = {"run": RUN_NAME, "questions": len(sample), "cases": CASE_LIMIT if RUN_CASES else 0, "models": {}}

for m in MODELS:
    out = model_dir(m["tag"])
    traces = sorted((out / "llm_trace").glob("*.jsonl"))
    if traces:
        files = " ".join(f'"{p}"' for p in traces)
        sh(f'{PY} scripts/llm_trace_report.py {files} > "{out / "reasoning_report.md"}"', check=False)
        sh(f'{PY} scripts/llm_trace_report.py {files} --prompts > "{out / "reasoning_report_with_prompts.md"}"', check=False)
    answers = out / "bulk500" / "answers_rag.jsonl"
    judge_free = {}
    if answers.exists():
        sh(f'{PY} {EVAL / "score.py"} report --gold {EVAL / "gold.jsonl"} --answers "{answers}" '
           f'--json-out "{out / "judge_free_report.json"}" > "{out / "judge_free_report.txt"}"', check=False)
        if (out / "judge_free_report.json").exists():
            rows = json.load(open(out / "judge_free_report.json", encoding="utf-8"))["rows"]
            mcq = [r["score"] for r in rows if r["category"] == "mcq_bar"]
            labels = [r["label_correct"] for r in rows if "label_correct" in r]
            judge_free = {"multiple_choice": round(statistics.mean(mcq), 3) if mcq else None, "mcq_n": len(mcq),
                          "yes_no_label": round(statistics.mean(labels), 3) if labels else None, "yes_no_n": len(labels)}
    ans = load_jsonl(answers) if answers.exists() else []
    case_rows = load_jsonl(out / "cases" / "answers_rag.jsonl") if (out / "cases" / "answers_rag.jsonl").exists() else []
    calls = [json.loads(l) for p in traces for l in open(p, encoding="utf-8") if l.strip()]
    by_site = defaultdict(list)
    for c in calls:
        by_site[c.get("call_site")].append(c)
    summary["models"][m["tag"]] = {
        "answers": len(ans), "errors": sum(bool(a.get("error")) for a in ans),
        "empty": sum(not str(a.get("answer") or "").strip() for a in ans),
        "repairs": dict(Counter(r for a in ans for r in a.get("repairs", []))),
        "answers_with_other_script_words": sum(bool(FOREIGN.search(a.get("answer") or "")) for a in ans),
        "cases": len(case_rows), "case_errors": sum(bool(c.get("error")) for c in case_rows),
        "judge_free": judge_free,
        "calls": {site: {"n": len(cs), "stopped_at_length": sum(c.get("done_reason") == "length" for c in cs),
                         "errors": sum(bool(c.get("error")) for c in cs),
                         "median_s": round(statistics.median([c.get("seconds") or 0 for c in cs]), 1),
                         "max_prompt_tokens": max((c.get("prompt_tokens") or 0) for c in cs),
                         "max_completion_tokens": max((c.get("completion_tokens") or 0) for c in cs)}
                  for site, cs in sorted(by_site.items())},
    }

(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
lines = [f"# Colab eval run {RUN_NAME}", "", f"{summary['questions']} questions, {summary['cases']} cases", ""]
for tag, s in summary["models"].items():
    jf = s["judge_free"]
    lines += [f"## {tag}", "",
              f"- answers {s['answers']} (errors {s['errors']}, empty {s['empty']}), cases {s['cases']} (errors {s['case_errors']})",
              f"- judge-free: multiple choice {jf.get('multiple_choice')} (n={jf.get('mcq_n')}), yes/no label {jf.get('yes_no_label')} (n={jf.get('yes_no_n')})",
              f"- repairs {s['repairs']}; answers with other-script words: {s['answers_with_other_script_words']}", "",
              "| call site | calls | stopped at length | errors | median s | max prompt tok | max output tok |",
              "|---|---|---|---|---|---|---|"]
    lines += [f"| {site} | {c['n']} | {c['stopped_at_length']} | {c['errors']} | {c['median_s']} | {c['max_prompt_tokens']} | {c['max_completion_tokens']} |"
              for site, c in s["calls"].items()] + [""]
(RUN_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## 8. Environment record and the results zip (on Drive)

In [ ]:
env_lines = []
for label, cmd in [("git", "git log -1 --format='%H %s'"), ("gpu", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"),
                   ("ollama", "ollama --version"), ("models", "ollama list"),
                   ("python", f"{PY} -c \"import sys, chromadb, sentence_transformers, torch; print(sys.version); print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__, '| torch', torch.__version__)\"")]:
    res = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    env_lines += [f"## {label}", (res.stdout + res.stderr).strip(), ""]
env_lines += ["## settings", json.dumps({k: globals()[k] for k in ("BRANCH", "MODELS", "LIMIT", "SAMPLE_SEED", "RUN_CASES", "CASE_LIMIT",
              "TOP_K", "CASE_TOP_K", "CONTEXT_LENGTH", "ANSWER_MAX_TOKENS", "CASE_MAX_TOKENS", "REQUEST_TIMEOUT_S")}, indent=1),
              "", "## corpus build", json.dumps(BUILD_INFO, ensure_ascii=False, indent=1)]
(RUN_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")

archive = shutil.make_archive(str(pathlib.Path(RESULTS_DIR) / f"{RUN_NAME}_results"), "zip", RUN_DIR)
print("results zip:", archive, f"({pathlib.Path(archive).stat().st_size / 1e6:.1f} MB)")
print("Send this zip back for the judging and the reasoning review.")